# Hessian Curvature Lens

Chapter 11: Matrix Calculus · *Mathematics for AI and Machine Learning* by Fuheng Wu

Inspect eigenvalues of Hessian to classify local curvature and conditioning.

Companion code for the [interactive demo](https://math4ai.org/companion/chapter-11/hessian-curvature/) on math4ai.org (sign-in required). Run the cell with Shift+Enter; PyTorch is preinstalled on Colab. Change the values at the top and run it again to explore.


In [1]:
import torch

# The demo's defaults: f(x, y) = 0.5 (a x^2 + 2 c x y + b y^2)
a, b, c = 2.0, 1.0, 0.4


def f(p):
    x, y = p
    return 0.5 * (a * x ** 2 + 2 * c * x * y + b * y ** 2)


# ── The Hessian by autograd equals [[a, c], [c, b]] ─────────────────────────
H = torch.autograd.functional.hessian(f, torch.tensor([0.3, -0.7], dtype=torch.float64))
print("Hessian =", H.tolist())
evals, evecs = torch.linalg.eigh(H)
print("eigenvalues =", evals.tolist())


def classify(evals):
    if (evals > 0).all():
        return "local minimum (bowl)"
    if (evals < 0).all():
        return "local maximum (cap)"
    return "saddle"


print("critical point at 0 is a", classify(evals))
print("condition number lambda_max / lambda_min =", (evals.max() / evals.min()).item())

# Curvature along a unit direction d is d^T H d, between the eigenvalues
for d in (torch.tensor([1.0, 0.0]), torch.tensor([0.0, 1.0]), evecs[:, 0], evecs[:, 1]):
    d = d.double() / d.norm()
    print(f"curvature along {[round(t, 3) for t in d.tolist()]}: {(d @ H @ d).item():.4f}")

# ── Flip signs to see the other cases ───────────────────────────────────────
for a_, b_, c_ in ((2.0, 1.0, 0.4), (2.0, -1.0, 0.4), (-2.0, -1.0, 0.4), (1.0, 1.0, 1.5)):
    ev = torch.linalg.eigvalsh(torch.tensor([[a_, c_], [c_, b_]], dtype=torch.float64))
    print(f"a={a_:+.1f} b={b_:+.1f} c={c_:.1f}: eigenvalues {[round(t, 3) for t in ev.tolist()]} -> {classify(ev)}")

Hessian = [[2.0, 0.4], [0.4, 1.0]]
eigenvalues = [0.8596875762567152, 2.1403124237432847]
critical point at 0 is a local minimum (bowl)
condition number lambda_max / lambda_min = 2.489639821320573
curvature along [1.0, 0.0]: 2.0000
curvature along [0.0, 1.0]: 1.0000
curvature along [0.331, -0.944]: 0.8597
curvature along [-0.944, -0.331]: 2.1403
a=+2.0 b=+1.0 c=0.4: eigenvalues [0.86, 2.14] -> local minimum (bowl)
a=+2.0 b=-1.0 c=0.4: eigenvalues [-1.052, 2.052] -> saddle
a=-2.0 b=-1.0 c=0.4: eigenvalues [-2.14, -0.86] -> local maximum (cap)
a=+1.0 b=+1.0 c=1.5: eigenvalues [-0.5, 2.5] -> saddle
